In [1]:
# 01 - Токенизатор

In [2]:
import json
import re
import torch
from torch.utils.data import Dataset, DataLoader

class MathTokenizer:
    def __init__(self):
        # Строго 19 токенов согласно конфигурации: 10 цифр + 4 спецсимвола + 4 CoT + 1 pad
        self.vocab = [str(i) for i in range(10)] + ["+", "-", "=", " ", "<c0>", "<c1>", "<b0>", "<b1>", "<pad>"]
        
        self.v2i = {v: i for i, v in enumerate(self.vocab)}
        self.i2v = {i: v for i, v in enumerate(self.vocab)}
        
        self.pad_token_id = self.v2i["<pad>"]
        self.vocab_size = len(self.vocab)

    def encode(self, text: str, from_split: bool = True) -> list[int]:
        if from_split:
            tokens = text.split()
        else:
            tokens = re.findall(r'<[cb][01]>|<pad>|\d|\+|-|=| ', text)
            
        return [self.v2i[t] for t in tokens]

    def decode(self, indices: list[int] | torch.Tensor) -> str:
        if isinstance(indices, torch.Tensor):
            indices = indices.tolist()
        return " ".join([self.i2v[i] for i in indices])

# Тест токенизатора
tokenizer = MathTokenizer()
print(f"Размер словаря: {tokenizer.vocab_size}")
assert tokenizer.vocab_size == 19

Размер словаря: 19


In [3]:
# 02 - загрузка данных

In [4]:
class MathCoTDataset(Dataset):
    def __init__(self, jsonl_path: str, tokenizer: MathTokenizer):
        self.tokenizer = tokenizer
        self.data = []
        
        print(f"Загрузка данных из {jsonl_path}...")
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                record = json.loads(line)
                tokens = self.tokenizer.encode(record["text"], from_split=True)
                self.data.append(torch.tensor(tokens, dtype=torch.long))
                
        print(f"Успешно загружено примеров: {len(self.data)}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]
        
train_dataset = MathCoTDataset("../data/train.jsonl", tokenizer)
test_dataset = MathCoTDataset("../data/test.jsonl", tokenizer)

train_loader = DataLoader(
    train_dataset, 
    batch_size=128, 
    shuffle=True,  
    num_workers=0, 
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=128, 
    shuffle=False, 
    pin_memory=True if torch.cuda.is_available() else False
)

# Проверка батча
batch = next(iter(train_loader))
print(f"Форма тренировочного батча: {batch.shape}")

Загрузка данных из ../data/train.jsonl...
Успешно загружено примеров: 240000
Загрузка данных из ../data/test.jsonl...
Успешно загружено примеров: 10000
Форма тренировочного батча: torch.Size([128, 24])


In [5]:
# 03 - инициализация модели и оптимизатора

In [8]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR

# Импортируем конфигурации
from grokking_carries.config import ModelConfig, TrainConfig
from grokking_carries.model.transformer import GrokkingCarriesTransformer

# Загрузка конфигурации и инициализация модели
model_cfg = ModelConfig()
train_cfg = TrainConfig() 

model = GrokkingCarriesTransformer(model_cfg).to(model_cfg.device)

# Разделение параметров для оптимизатора AdamW
decay_params = []
nodecay_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if param.dim() < 2 or 'ln' in name:
        nodecay_params.append(param)
    else:
        decay_params.append(param)

optim_groups = [
    {'params': decay_params, 'weight_decay': train_cfg.weight_decay}, 
    {'params': nodecay_params, 'weight_decay': 0.0}
]

optimizer = AdamW(optim_groups, lr=train_cfg.lr, betas=(0.9, 0.95))

# Конфигурация расписания Learning Rate (Warmup + Cosine Decay)
epochs = 100  
total_steps = len(train_loader) * epochs

warmup_steps = train_cfg.warmup_steps  

warmup_scheduler = LinearLR(
    optimizer, 
    start_factor=1e-8, 
    end_factor=1.0, 
    total_iters=warmup_steps
)

cosine_scheduler = CosineAnnealingLR(
    optimizer, 
    T_max=(total_steps - warmup_steps), 
    eta_min=1e-6
)

scheduler = SequentialLR(
    optimizer, 
    schedulers=[warmup_scheduler, cosine_scheduler], 
    milestones=[warmup_steps]
)

print(f"Модель успешно импортирована из пакета grokking_carries и развернута на: {model_cfg.device}")
print(f"Общее количество параметров архитектуры: {sum(p.numel() for p in model.parameters()):,}")

Модель успешно импортирована из пакета grokking_carries и развернута на: cpu
Общее количество параметров архитектуры: 3,181,568


In [9]:
# 04 - запуск обучения

In [10]:
import time
import torch
import torch.nn as nn

criterion = nn.CrossEntropyLoss(ignore_index=-100)

eq_token_id = tokenizer.v2i["="] 

epochs = 100
history = {'train_loss': [], 'val_loss': []}

print("Старт обучения...")
start_total_time = time.time()

for epoch in range(1, epochs + 1):
    epoch_start_time = time.time()
    
    model.train()
    train_loss = 0.0
    
    for batch in train_loader:
        batch = batch.to(model_cfg.device)
        
        logits = model(batch)
        shift_logits = logits[:, :-1, :].contiguous()
        
        shift_targets = batch[:, 1:].clone().contiguous()

        shift_targets[shift_targets == model_cfg.pad_token_id] = -100
        
        # Ищем знак '=' и маскируем всё, что было до него
        for i in range(batch.size(0)):
            eq_pos = (batch[i] == eq_token_id).nonzero(as_tuple=True)[0]
            if len(eq_pos) > 0:

                shift_targets[i, :eq_pos[0]] = -100
                
        # Лосс считается ТОЛЬКО по CoT-токенам и финальному ответу
        loss = criterion(shift_logits.view(-1, model_cfg.vocab_size), shift_targets.view(-1))
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        
        train_loss += loss.item()
        
    avg_train_loss = train_loss / len(train_loader)
    history['train_loss'].append(avg_train_loss)
    
    # Валидация
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(model_cfg.device)
            logits = model(batch)
            
            shift_logits = logits[:, :-1, :].contiguous()
            shift_targets = batch[:, 1:].clone().contiguous()
            
            # Применяем то же самое маскирование для чистой валидации
            shift_targets[shift_targets == model_cfg.pad_token_id] = -100
            for i in range(batch.size(0)):
                eq_pos = (batch[i] == eq_token_id).nonzero(as_tuple=True)[0]
                if len(eq_pos) > 0:
                    shift_targets[i, :eq_pos[0]] = -100
                    
            loss = criterion(shift_logits.view(-1, model_cfg.vocab_size), shift_targets.view(-1))
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(test_loader)
    history['val_loss'].append(avg_val_loss)
    
    current_lr = scheduler.get_last_lr()[0]
    epoch_time = time.time() - epoch_start_time
    
    print(f"Epoch {epoch:03d}/{epochs} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"LR: {current_lr:.2e} | Time: {epoch_time:.1f}s")

total_time = time.time() - start_total_time
print(f"Обучение завершено за {total_time:.1f} секунд.")

Старт обучения...
Epoch 001/100 | Train Loss: 0.1496 | Val Loss: 0.0129 | LR: 1.00e-03 | Time: 470.9s
Epoch 002/100 | Train Loss: 0.0026 | Val Loss: 0.0029 | LR: 9.99e-04 | Time: 468.9s
Epoch 003/100 | Train Loss: 0.0016 | Val Loss: 0.0025 | LR: 9.98e-04 | Time: 493.0s
Epoch 004/100 | Train Loss: 0.0008 | Val Loss: 0.0000 | LR: 9.97e-04 | Time: 511.2s


KeyboardInterrupt: 

In [13]:
import torch

def generate_answer(model, tokenizer, prompt_text: str, max_new_tokens: int = 10, device: str = 'cpu') -> str:
    """Генерирует CoT-ответ модели авторегрессионно, токен за токеном."""
    model.eval()
    
    input_ids = tokenizer.encode(prompt_text, from_split=True)
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)
    
    generated = input_tensor
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(generated)
            next_token_logits = logits[:, -1, :]
            next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token_id], dim=1)
            
            if next_token_id.item() == tokenizer.pad_token_id:
                break
                
    generated_ids = generated[0].tolist()
    prompt_len = len(input_ids)
    answer_ids = generated_ids[prompt_len:]
    
    if tokenizer.pad_token_id in answer_ids:
        answer_ids = answer_ids[:answer_ids.index(tokenizer.pad_token_id)]
        
    return tokenizer.decode(answer_ids).strip()

# ТЕСТИРОВАНИЕ НА РАЗНЫХ ПРИМЕРАХ

prompt_easy = "0 0 0 4 5 + 0 0 0 9 2 ="
ans_easy = generate_answer(model, tokenizer, prompt_easy, device=model_cfg.device)

print("--- Базовый тест ---")
print(f"Промпт: {prompt_easy}")
print(f"Вывод:  {ans_easy}")
print(f"Ожид.:  <c0> 7 <c1> 3 <c0> 1 <c0> 0 <c0> 0\n")

prompt_hard = "0 9 9 9 9 + 0 0 0 0 1 ="
ans_hard = generate_answer(model, tokenizer, prompt_hard, device=model_cfg.device)

print("--- Тест OOD (Каскадный перенос) ---")
print(f"Промпт: {prompt_hard}")
print(f"Вывод:  {ans_hard}")
print(f"Ожид.:  <c1> 0 <c1> 0 <c1> 0 <c1> 0 <c0> 1\n")

prompt_sub = "0 0 5 0 2 - 0 0 0 1 9 ="
ans_sub = generate_answer(model, tokenizer, prompt_sub, device=model_cfg.device)

print("--- Тест вычитания ---")
print(f"Промпт: {prompt_sub}")
print(f"Вывод:  {ans_sub}")
print(f"Ожид.:  <b1> 3 <b1> 8 <b0> 4 <b0> 0 <b0> 0")

--- Базовый тест ---
Промпт: 0 0 0 4 5 + 0 0 0 9 2 =
Вывод:  <c0> 7 <c1> 3 <c0> 1 <c0> 0 <c0> 0
Ожид.:  <c0> 7 <c1> 3 <c0> 1 <c0> 0 <c0> 0

--- Тест OOD (Каскадный перенос) ---
Промпт: 0 9 9 9 9 + 0 0 0 0 1 =
Вывод:  <c1> 0 <c1> 0 <c1> 0 <c1> 0 <c0> 1
Ожид.:  <c1> 0 <c1> 0 <c1> 0 <c1> 0 <c0> 1

--- Тест вычитания ---
Промпт: 0 0 5 0 2 - 0 0 0 1 9 =
Вывод:  <b1> 3 <b1> 8 <b0> 4 <b0> 0 <b0> 0
Ожид.:  <b1> 3 <b1> 8 <b0> 4 <b0> 0 <b0> 0
